In [133]:
import sys
sys.path.append("../")

In [134]:
import torch
from qmpsqsc.models import mpsqsc
from qmpsqsc.models import qmps
from importlib import reload
from qmpsqsc.models.data.utils import flip_sites_in_mps
import qmpsqsc.models.data as mpsdata

reload(mpsqsc)

<module 'qmpsqsc.models.mpsqsc' from '/Users/keisuke/Documents/projects/mps4qsc/notebooks/../qmpsqsc/models/mpsqsc/__init__.py'>

In [135]:
import torch.nn.functional as F

L = 30
chi = 2
d = 2
ghz = mpsqsc.build_ghz_state(L, d, chi).to(dtype=torch.complex128)
ghz = ghz.normalize()

zghz = ghz.copy()
zghz.As[0][:, 1] = -ghz.As[0][:, 1]

ghz_X_errors = [flip_sites_in_mps(ghz, [i], "X") for i in range(L)]
ghz_Y_errors = [flip_sites_in_mps(ghz, [i], "Y") for i in range(L)]
ghz_Z_errors = [flip_sites_in_mps(ghz, [i], "Z") for i in range(L)]

zghz_X_errors = [flip_sites_in_mps(zghz, [i], "X") for i in range(L)]
zghz_Y_errors = [flip_sites_in_mps(zghz, [i], "Y") for i in range(L)]
zghz_Z_errors = [flip_sites_in_mps(zghz, [i], "Z") for i in range(L)]

allup = mpsqsc.build_classical_state(L, d, [0]*L).to(dtype=torch.complex128)
alldown = mpsqsc.build_classical_state(L, d, [1]*L).to(dtype=torch.complex128)


In [136]:
ghzs_X = mpsqsc.add_mpstates(ghz_X_errors)
ghzs_Y = mpsqsc.add_mpstates(ghz_Y_errors)



zghzs_X = mpsqsc.add_mpstates(zghz_X_errors)
zghzs_Y = mpsqsc.add_mpstates(zghz_Y_errors)


In [137]:
from qmpsqsc.models.mpsqsc.compress import compress_mpstate

ghzs_X, ghzsX_fid = compress_mpstate(ghzs_X, 4, adam_steps=0, n_sweeps=1)
ghzs_Y, ghzsY_fid = compress_mpstate(ghzs_Y, 4, adam_steps=0, n_sweeps=1)

print("fidelities", ghzsX_fid, ghzsY_fid)

zghzs_X, zghzX_fid = compress_mpstate(zghzs_X, 4, adam_steps=0, n_sweeps=1)
zghzs_Y, zghzY_fid = compress_mpstate(zghzs_Y, 4, adam_steps=0, n_sweeps=1)

print("fidelities", zghzX_fid, zghzY_fid)


ghzs = mpsqsc.add_mpstates([ghz, ghzs_X])
ghzs = ghzs.normalize()

ghzs, ghzs_fid = compress_mpstate(ghzs, 6, adam_steps=0, n_sweeps=1)

print("fidelities", ghzs_fid)

zghzs = mpsqsc.add_mpstates([zghz, zghzs_X])
zghzs = zghzs.normalize()

zghzs, zghz_fid = compress_mpstate(zghzs, 6, adam_steps=0, n_sweeps=1)

print("fidelities", zghz_fid)


fidelities (1.0000000000000022+0j) (1.0000000000000042-1.9121355040559703e-16j)
fidelities (1.000000000000004+0j) (1.0000000000000038-7.442135430798189e-16j)
fidelities (1.0000000000000004+0j)
fidelities (1.000000000000001+0j)


In [ ]:
import torch
import torch.nn as nn
import math

class LinearSVM(nn.Module):
    def __init__(self, in_dim: int, dtype: torch.dtype = torch.float64):
        super().__init__()
        self.linear = nn.Linear(in_dim, 1, dtype=dtype)  # w^T phi(x) + b

    def forward(self, x):
        # x: (B, in_dim)
        scores = self.linear(x).squeeze(-1)  # (B,)
        return scores


def svm_hinge_loss(model: LinearSVM,
                   scores: torch.Tensor,
                   y: torch.Tensor,
                   C: float = 1.0):
    """
    scores: (B,) = f(x)
    y: (B,) with values 0 or 1
    """
    # 0/1 -> -1/+1
    y_pm1 = 2 * y.float() - 1.0

    margins = 2 - y_pm1 * scores
    hinge = torch.clamp(margins, min=0.0).mean()

    # L2 regularization on w
    w = model.linear.weight
    l2_reg = 0.002 * torch.sum(w * w)

    return l2_reg + C * hinge


def poly2_features(V: torch.Tensor) -> torch.Tensor:
    """
    V: (B, 2) with columns [x1, x2]
    Returns phi(V): (B, 3) for degree-2 homogeneous poly kernel:
        k(x, z) = (x^T z)^2
    Feature map: [x1^2, sqrt(2)*x1*x2, x2^2]
    """
    x1 = V[:, 0:1]
    x2 = V[:, 1:1+1]

    phi = torch.cat(
        [
            x1,
            x2,
            x1 ** 2,
            math.sqrt(2.0) * x1 * x2,
            x2 ** 2,
        ],
        dim=1,
    )
    return phi

def svm_accuracy(scores, y):
    """
    scores: (B,) SVM outputs
    y: (B,) true labels in {0,1}
    """
    # Predict
    pred_pm1 = torch.sign(scores)         # -1 or +1
    pred = (pred_pm1 > 0).long()          # 0 or 1

    # Compare with labels
    correct = (pred == y).float().sum()
    acc = correct / y.numel()
    return acc

In [160]:
svm = LinearSVM(in_dim=5, dtype=torch.float64)  # because phi(V) has dim 3
svm.train()

LinearSVM(
  (linear): Linear(in_features=5, out_features=1, bias=True)
)

In [161]:
data_generator = mpsdata.ghz.create_ghz_rho_batch_qsc(ghz, allup, alldown, 2**6, 0.5, random_flip=True)

In [164]:
ghzs.set_requires_grad(True)
zghzs.set_requires_grad(True)
optimizer = torch.optim.Adam(ghzs.As + zghzs.As, lr=0.001)
optim_svm = torch.optim.Adam(svm.parameters(), lr=0.01)

optimizer.zero_grad()
optim_svm.zero_grad()
svm_train_steps = 30

for _ in range(1000):
    states, labels, errors = next(data_generator)
    # loss, acc = mpsdata.calculate_loss_mpstates(ghzs, zghzs, states, labels)
    probs, norms = mpsdata.mps_binary_predict(ghzs, zghzs, states)
    phi_V = poly2_features(probs)  # (B, 3)
    for _ in range(svm_train_steps):
        optimizer.zero_grad()
        optim_svm.zero_grad()
        scores = svm(phi_V.detach().clone())        # (B,)
        loss = svm_hinge_loss(svm, scores, labels, C=0.5)
        loss.backward()
        acc = svm_accuracy(scores, labels)
        optim_svm.step()
        # print("internal loss", loss.item(), acc.mean().item())
    optimizer.zero_grad()
    scores = svm(phi_V)        # (B,)
    loss = svm_hinge_loss(svm, scores, labels, C=0.5)
    optimizer.step()
    acc = svm_accuracy(scores, labels)
    print(loss.item(), acc.mean().item())


0.133200962949882 1.0
0.1335838631779694 1.0
0.1329858263671723 1.0
0.133160935874004 1.0
0.1324589112996933 1.0
0.13310241253206032 1.0
0.13204045886782959 1.0
0.13243401435623658 1.0
0.1323558493302938 1.0
0.13208392143304795 1.0
0.1321608859618449 1.0
0.13173060573636652 1.0
0.1317273197291464 1.0
0.13141145718803074 1.0
0.1315127751430879 1.0
0.1321881219178432 1.0
0.13150797183754673 1.0
0.13212978637350864 1.0
0.1316454218539028 1.0
0.13089059888684568 1.0
0.1308715814482956 1.0
0.13098484242600805 1.0
0.1309321290853276 1.0
0.1309284184926925 1.0
0.13156953572921373 1.0
0.1305269017114221 1.0


KeyboardInterrupt: 

<module 'qmpsqsc.models.mpsqsc.helper' from '/Users/keisuke/Documents/projects/mps4qsc/notebooks/../qmpsqsc/models/mpsqsc/helper.py'>

In [165]:
mps_qsc = mpsqsc.helper.build_qsc_from_mpstates(ghzs, zghzs)
mps_qsc = mps_qsc.truncate_bond_dimension(2)
mps_qsc.set_requires_grad(True)

In [185]:
optimizer = torch.optim.Adam(mps_qsc.As, lr=0.001)
optim_svm = torch.optim.Adam(svm.parameters(), lr=0.01)

optimizer.zero_grad()
optim_svm.zero_grad()

svm_train_steps = 30  # set number of SVM training steps per outer loop

for _ in range(1000):
    states, labels, _ = next(data_generator)
    probs, norms = mps_qsc.predict(states)
    phi_V = poly2_features(probs)  # (B, 3)
    
    # Train SVM multiple times on this minibatch (internal loop)
    for _ in range(svm_train_steps):
        optim_svm.zero_grad()
        # SVM input should not propagate gradient to mps_qsc
        scores = svm(phi_V.detach().clone())
        loss_svm = svm_hinge_loss(svm, scores, labels, C=0.5)
        loss_svm.backward()
        optim_svm.step()
    
    # Now update mps_qsc using SVM output (stop SVM gradient!)
    optimizer.zero_grad()
    scores = svm(phi_V)
    loss = svm_hinge_loss(svm, scores, labels, C=0.5)
    loss.backward()
    optimizer.step()
    optim_svm.zero_grad()
    optimizer.zero_grad()
    acc = svm_accuracy(scores, labels)
    print(loss.item(), acc.mean().item())


0.13190535405996634 1.0
0.12833536526775574 1.0
0.1308030155884015 1.0
0.1287878114710289 1.0
0.13067346193680232 1.0
0.1307716892010334 1.0
0.1307082068215933 1.0
0.12875324341271024 1.0
0.12841465187622966 1.0
0.12862070734930575 1.0
0.1285653951093232 1.0
0.13142365065697217 1.0
0.12912924175986817 1.0
0.13133681662691338 1.0
0.12853593743590852 1.0
0.12841736811884846 1.0
0.13144882532663785 1.0
0.13391664451253174 1.0
0.12847861810038746 1.0
0.1288304662028473 1.0
0.13145624027967187 1.0
0.1312543359094358 1.0
0.128665155119676 1.0
0.13070545500484604 1.0
0.12872823383669665 1.0
0.1282758452140946 1.0
0.1284645129331369 1.0
0.13171798571365032 1.0
0.13375262935169088 1.0
0.12836697013776566 1.0
0.12862104018713022 1.0
0.13129878090622948 1.0
0.12846528327448692 1.0
0.12865078930127855 1.0
0.12878043977874892 1.0
0.12924025871881023 1.0
0.12920279611500896 1.0
0.13046924152854258 1.0
0.12840544829458095 1.0
0.13146288592101316 1.0
0.12846552089559438 1.0
0.130908659231581 1.0
0.128

KeyboardInterrupt: 

In [186]:
mps_qsc = mps_qsc.canonicalize(truncate=True)
Us, last = qmps.construct_unitary_from_As(mps_qsc.As)
qmps_ghz = qmps.qMPS(L, chi, d, Us=Us, last_unitary=last)

In [199]:
svm_qmps = LinearSVM(in_dim=5, dtype=torch.float64)
svm_qmps.linear.weight.data[:] = svm.linear.weight.data.clone()

In [200]:
svm(phi_V)

tensor([-1.9969, -1.9969, -1.9997,  2.0027, -1.9969,  2.0028, -1.9969, -1.9969,
        -1.9969,  2.0027, -1.9969, -1.9969,  2.0028, -1.9969, -1.9374, -1.9997,
        -1.6952, -1.9969,  2.0027, -1.8213,  2.0028, -1.9969, -1.9969,  2.0027,
        -1.9858,  2.0028, -1.9969,  2.0027, -1.9969,  2.0027,  2.0027,  2.0028,
         2.0027, -1.9969, -1.9969, -1.9374, -1.9969, -1.9969,  2.0027,  2.0028,
         2.0027, -1.9459,  2.0001,  2.0028, -1.9969, -1.9969, -1.9997,  2.0028,
         2.0028,  2.0028, -1.6706, -1.9997,  2.0027,  2.0028,  2.0027,  2.0028,
         2.0028,  2.0028,  2.0028,  2.0027,  2.0027,  2.0027,  2.0027, -1.8782],
       dtype=torch.float64, grad_fn=<SqueezeBackward1>)

In [201]:
svm_qmps(phi_V)

tensor([-4.2986, -4.2986, -4.3015, -0.2991, -4.2986, -0.2990, -4.2986, -4.2986,
        -4.2986, -0.2991, -4.2986, -4.2986, -0.2990, -4.2986, -4.2392, -4.3015,
        -3.9969, -4.2986, -0.2991, -4.1230, -0.2990, -4.2986, -4.2986, -0.2991,
        -4.2876, -0.2990, -4.2986, -0.2991, -4.2986, -0.2991, -0.2991, -0.2990,
        -0.2991, -4.2986, -4.2986, -4.2392, -4.2986, -4.2986, -0.2991, -0.2990,
        -0.2991, -4.2477, -0.3016, -0.2990, -4.2986, -4.2986, -4.3015, -0.2990,
        -0.2990, -0.2990, -3.9723, -4.3015, -0.2991, -0.2990, -0.2991, -0.2990,
        -0.2990, -0.2990, -0.2990, -0.2991, -0.2991, -0.2991, -0.2991, -4.1799],
       dtype=torch.float64, grad_fn=<SqueezeBackward1>)

In [192]:
w = 0.001
qmps_ghz.set_weights(w)

probs, norms = qmps_ghz.predict(states)
phi_V = poly2_features(probs)  # (B, 3)
scores = svm_qmps(phi_V)        # (B,)
loss = svm_hinge_loss(svm_qmps, scores, labels, C=0.5)
print(loss.item())

0.6924435078746023


In [189]:
from qmpsqsc.models.qmps.optimizer import StiefelAdam

optimizer = StiefelAdam(qmps_ghz.unitaries(), lr=0.001)
optim_svm = torch.optim.Adam(svm_qmps.parameters(), lr=0.001)
svm_train_steps = 5  # or any number you had before

for step in range(1000):
    states, labels, _ = next(data_generator)
    optimizer.zero_grad(set_to_none=True)

    # SVM internal training loop (train SVM multiple times per outer step)
    for _ in range(svm_train_steps):
        optim_svm.zero_grad()
        probs, norms = qmps_ghz.predict(states)
        phi_V = poly2_features(probs)
        scores = svm_qmps(phi_V.detach().clone())
        loss_svm = svm_hinge_loss(svm_qmps, scores, labels, C=0.5)
        loss_svm.backward()
        optim_svm.step()

    # Now use the SVM output in the loss, propagate grad ONLY to qmps_ghz/unitaries
    probs, norms = qmps_ghz.predict(states)
    phi_V = poly2_features(probs)
    scores = svm_qmps(phi_V)
    loss = svm_hinge_loss(svm_qmps, scores, labels, C=0.5)
    loss.backward()
    optimizer.step()
    optim_svm.zero_grad()
    optimizer.zero_grad()

    acc = (scores.argmax(dim=-1) == labels).float()

    print(f"[step {step:5d}] loss={loss.item():.6f}  acc={acc.mean().item():.3f}")


[step     0] loss=0.702402  acc=0.500
[step     1] loss=0.699060  acc=0.000


KeyboardInterrupt: 